In [ ]:
#ritnet_lite.ipynb  (paste into Colab, one cell or cell by cell, GPU runtime)
# ---------------------------------------------------------------
# Trains thinner pupil-segmentation networks, compares them with your existing
# models on the numbers the tracker actually uses (Dice, pupil diameter error in
# px, missed frames), and exports ONNX files for the Jetson.
#
# What is different from the previous notebook:
#   * New network (LitePupilNet): runs its deep layers at half resolution with
#     cheap additive skips, then a small full-resolution head sharpens the edge.
#     1.3 GFLOPs (lite) and 0.7 GFLOPs (tiny) at 400x400, against 3.7 for the
#     thin model and 14.7 for the original.
#   * Augmentation (the old notebook trained on raw images only): small
#     rotation/zoom/shift, horizontal flip, brightness/contrast/gamma, noise.
#   * Boundary-weighted loss, because the app measures the edge of the pupil.
#   * Distillation from your existing models when their .pth files are found.
#   * Best epoch chosen by validation Dice, not validation loss.
#   * ONNX export at the end. Build the engine from the .onnx on the Jetson.
# ---------------------------------------------------------------

import os
import sys
import math
import time
import zipfile
import subprocess
from collections import Counter

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

# ======================= SETTINGS (edit these) =================
DRIVE_DIR    = "/content/drive/MyDrive"
ZIP_PATH     = DRIVE_DIR + "/annotated.zip"
DATA_DIR     = "/content/dataset"
DATASET_ROOT = DATA_DIR + "/annotated"          # holds train/ val/ test/, each with images/ and masks/

IMG_SIZE     = 400                              # what the app feeds the network, divisible by 16

# name -> channel widths at 1/2, 1/4, 1/8, 1/16 resolution
VARIANTS     = {"lite": (12, 24, 48, 96),       # ~1.3 GFLOPs
                "tiny": (8, 16, 32, 64)}        # ~0.7 GFLOPs   (add "plus": (16, 32, 64, 128) for ~2.0)
TRAIN_THESE  = ["lite", "tiny"]

EPOCHS       = 40
BATCH_SIZE   = 16
LR           = 2e-3
WEIGHT_DECAY = 1e-4
BOUNDARY_W   = 3.0                              # extra loss weight on pixels at the pupil edge
KD_WEIGHT    = 0.5                              # distillation strength, used only if teachers exist

# one folder for every .pth and .onnx (relative paths resolve from the notebook's working folder)
WEIGHTS_DIR  = "/content/drive/MyDrive/weights"   # absolute path: My Drive/weights2

# existing models, used as teachers and as baselines in the final table
OLD_WEIGHTS  = WEIGHTS_DIR + "/pupil_tracker_best.pth"  # widths (16, 32, 64, 128, 256)
THIN_WEIGHTS = WEIGHTS_DIR + "/pupil_tracker_thin.pth"  # widths (8, 16, 32, 64, 128)

# augmentation strength
ROT_DEG, ZOOM, SHIFT, FLIP_H = 12.0, 0.10, 0.06, True
# ===============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
print("Using device:", DEVICE)

# ---------------------------------------------------------
# 1. DRIVE, PACKAGES, DATASET
# ---------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# create the folder only AFTER Drive is mounted, otherwise it lands on Colab's temporary disk
os.makedirs(WEIGHTS_DIR, exist_ok=True)
WEIGHTS_DIR = os.path.abspath(WEIGHTS_DIR)
print("Weights and ONNX folder (full path):", WEIGHTS_DIR)
if IN_COLAB and not os.path.ismount("/content/drive"):
    print("WARNING: Google Drive is not mounted, files will NOT appear in My Drive")

try:
    import onnx
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"])
    import onnx

if not os.path.isdir(DATASET_ROOT):
    print("Extracting dataset...")
    zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
    print("Extraction complete.")

import shutil
if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
print("Cleared. Re-extracting from zip...")
zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
print("Done.")

def load_split(split):
    """Reads a whole split into uint8 tensors (N,1,H,W): images and binary masks."""
    img_dir = os.path.join(DATASET_ROOT, split, "images")
    mask_dir = os.path.join(DATASET_ROOT, split, "masks")
    names = sorted(n for n in os.listdir(img_dir) if n.lower().endswith((".jpg", ".jpeg", ".png")))
    xs, ys = [], []
    for n in names:
        stem = os.path.splitext(n)[0]
        xs.append(np.array(Image.open(os.path.join(img_dir, n)).convert("L"), dtype=np.uint8))
        m = np.array(Image.open(os.path.join(mask_dir, stem + "_mask.png")).convert("L"), dtype=np.uint8)
        ys.append((m > 127).astype(np.uint8))
    try:
        x, y = np.stack(xs), np.stack(ys)
    except ValueError:
        sys.exit("Images in '%s' do not all have the same size: %s" % (split, sorted(set(a.shape for a in xs))))
    return torch.from_numpy(x)[:, None], torch.from_numpy(y)[:, None]


Xtr, Ytr = load_split("train")
Xva, Yva = load_split("val")
Xte, Yte = load_split("test")
print("train %d | val %d | test %d | image size %dx%d"
      % (len(Xtr), len(Xva), len(Xte), Xtr.shape[3], Xtr.shape[2]))
if Xtr.shape[2] != IMG_SIZE or Xtr.shape[3] != IMG_SIZE:
    print("WARNING: your images are not %dx%d, but the app feeds the network %dx%d. "
          "Check that the training images come from the same crop as the live frames."
          % (IMG_SIZE, IMG_SIZE, IMG_SIZE, IMG_SIZE))
if Xtr.shape[2] % 16 or Xtr.shape[3] % 16:
    sys.exit("Image width and height must be divisible by 16.")
if DEVICE == "cuda" and Xtr.numel() * 2 < 2.5e9:     # whole training set fits on the GPU: no loader needed
    Xtr, Ytr = Xtr.to(DEVICE), Ytr.to(DEVICE)

# ---------------------------------------------------------
# 2. NETWORKS
# ---------------------------------------------------------
def cbr(i, o, s=1):
    return nn.Sequential(nn.Conv2d(i, o, 3, stride=s, padding=1, bias=False),
                         nn.BatchNorm2d(o), nn.ReLU(inplace=True))


class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.conv = nn.Sequential(cbr(i, o), cbr(o, o))

    def forward(self, x):
        return self.conv(x)


class SlimUNet(nn.Module):
    """Your existing network. Kept so the old weights load as teachers and baselines."""
    def __init__(self, in_channels=1, out_channels=1, c=(16, 32, 64, 128, 256)):
        super().__init__()
        self.down1 = DoubleConv(in_channels, c[0])
        self.down2 = DoubleConv(c[0], c[1])
        self.down3 = DoubleConv(c[1], c[2])
        self.down4 = DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = DoubleConv(c[3], c[4])
        self.up1 = nn.ConvTranspose2d(c[4], c[3], 2, 2); self.up_conv1 = DoubleConv(c[4], c[3])
        self.up2 = nn.ConvTranspose2d(c[3], c[2], 2, 2); self.up_conv2 = DoubleConv(c[3], c[2])
        self.up3 = nn.ConvTranspose2d(c[2], c[1], 2, 2); self.up_conv3 = DoubleConv(c[2], c[1])
        self.up4 = nn.ConvTranspose2d(c[1], c[0], 2, 2); self.up_conv4 = DoubleConv(c[1], c[0])
        self.final_conv = nn.Conv2d(c[0], out_channels, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        d4 = self.down4(self.pool(d3))
        b = self.bottleneck(self.pool(d4))
        c1 = self.up_conv1(torch.cat((d4, self.up1(b)), 1))
        c2 = self.up_conv2(torch.cat((d3, self.up2(c1)), 1))
        c3 = self.up_conv3(torch.cat((d2, self.up3(c2)), 1))
        c4 = self.up_conv4(torch.cat((d1, self.up4(c3)), 1))
        return self.final_conv(c4)


class LitePupilNet(nn.Module):
    """Deep layers at 1/2, 1/4, 1/8, 1/16 resolution with additive skips (cheap),
    then a full-resolution head that sees the raw image again to place the edge.
    c = channel widths at 1/2, 1/4, 1/8, 1/16 resolution; d = full-res detail channels."""
    def __init__(self, c=(12, 24, 48, 96), d=4):
        super().__init__()
        self.detail = cbr(1, d)
        self.stem = cbr(1, c[0], s=2)
        self.e1 = cbr(c[0], c[0])
        self.e2 = DoubleConv(c[0], c[1])
        self.e3 = DoubleConv(c[1], c[2])
        self.e4 = DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2, 2)
        self.u3 = nn.ConvTranspose2d(c[3], c[2], 2, 2); self.d3 = cbr(c[2], c[2])
        self.u2 = nn.ConvTranspose2d(c[2], c[1], 2, 2); self.d2 = cbr(c[1], c[1])
        self.u1 = nn.ConvTranspose2d(c[1], c[0], 2, 2); self.d1 = cbr(c[0], c[0])
        self.u0 = nn.ConvTranspose2d(c[0], 8, 2, 2)
        self.head = nn.Sequential(cbr(8 + d, 8), nn.Conv2d(8, 1, 1))

    def forward(self, x):
        det = self.detail(x)
        s1 = self.e1(self.stem(x))
        s2 = self.e2(self.pool(s1))
        s3 = self.e3(self.pool(s2))
        b = self.e4(self.pool(s3))
        y = self.d3(self.u3(b) + s3)
        y = self.d2(self.u2(y) + s2)
        y = self.d1(self.u1(y) + s1)
        return self.head(torch.cat((self.u0(y), det), 1))


# existing models: teachers for distillation and baselines for the final table
baselines = []
for bname, bw, bpath in (("old", (16, 32, 64, 128, 256), OLD_WEIGHTS),
                         ("thin", (8, 16, 32, 64, 128), THIN_WEIGHTS)):
    if os.path.exists(bpath):
        m = SlimUNet(c=bw).to(DEVICE)
        m.load_state_dict(torch.load(bpath, map_location=DEVICE))
        m.eval()
        for p in m.parameters():
            p.requires_grad_(False)
        baselines.append((bname, m))
print("existing models found:", [b[0] for b in baselines] or "none (training without distillation)")

# ---------------------------------------------------------
# 3. AUGMENTATION, LOSS, METRICS
# ---------------------------------------------------------
def augment(x, y):
    """x, y: float (B,1,H,W) on DEVICE, x in 0..1, y in {0,1}. Same geometry for both."""
    B = x.shape[0]
    dev = x.device
    r = lambda: (torch.rand(B, device=dev) * 2 - 1)          # uniform in -1..1
    ang = r() * math.radians(ROT_DEG)
    sc = 1.0 + r() * ZOOM
    flip = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0) if FLIP_H else torch.ones(B, device=dev)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0] = torch.cos(ang) / sc * flip
    theta[:, 0, 1] = -torch.sin(ang) / sc
    theta[:, 0, 2] = r() * SHIFT * 2
    theta[:, 1, 0] = torch.sin(ang) / sc * flip
    theta[:, 1, 1] = torch.cos(ang) / sc
    theta[:, 1, 2] = r() * SHIFT * 2
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    y = (F.grid_sample(y, grid, mode="bilinear", padding_mode="zeros", align_corners=False) > 0.5).float()

    g = torch.exp(r().view(B, 1, 1, 1) * math.log(1.4))       # gamma 0.71 .. 1.4
    x = x.clamp(1e-4, 1.0) ** g
    mean = x.mean((2, 3), keepdim=True)
    x = (x - mean) * (1.0 + r().view(B, 1, 1, 1) * 0.3) + mean + r().view(B, 1, 1, 1) * 0.08
    x = x + torch.randn_like(x) * torch.rand(B, 1, 1, 1, device=dev) * 0.02
    return x.clamp(0, 1), y


def seg_loss(logits, y):
    """Edge-weighted BCE + per-image soft Dice."""
    edge = (F.max_pool2d(y, 5, 1, 2) + F.max_pool2d(1 - y, 5, 1, 2) - 1).clamp(0, 1)   # 1 within 2 px of the border
    w = 1.0 + BOUNDARY_W * edge
    bce = (F.binary_cross_entropy_with_logits(logits, y, reduction="none") * w).sum() / w.sum()
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
    return bce + dice.mean()


def fit_ellipse(mask):
    """Same pupil extraction as the tracker: largest contour over 100 px, then fitEllipse."""
    cs = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cs = cs[0] if len(cs) == 2 else cs[1]
    cs = [c for c in cs if len(c) >= 5 and cv2.contourArea(c) > 100]
    if not cs:
        return None
    (cx, cy), (MA, ma), _ = cv2.fitEllipse(max(cs, key=cv2.contourArea))
    return cx, cy, (MA + ma) / 2.0


def app_eval(net, X, Y):
    """Metrics on a split, computed the way the app uses the mask (logit > 0, as in the TensorRT path)."""
    net.eval()
    dice, iou, derr, cerr, missed, false_pos = [], [], [], [], 0, 0
    with torch.no_grad():
        for i in range(0, len(X), 16):
            xb = X[i:i + 16].to(DEVICE).float() / 255.0
            pred = (net(xb) > 0).cpu().numpy().astype(np.uint8)[:, 0]
            gt = Y[i:i + 16].cpu().numpy().astype(np.uint8)[:, 0]
            for p, g in zip(pred, gt):
                inter = float((p & g).sum())
                dice.append((2 * inter + 1) / (float(p.sum()) + float(g.sum()) + 1))
                iou.append((inter + 1) / (float((p | g).sum()) + 1))
                ep, eg = fit_ellipse(p), fit_ellipse(g)
                if eg is not None and ep is not None:
                    derr.append(abs(ep[2] - eg[2]))
                    cerr.append(math.hypot(ep[0] - eg[0], ep[1] - eg[1]))
                elif eg is not None:
                    missed += 1
                elif ep is not None:
                    false_pos += 1
    pct = lambda a, q: float(np.percentile(a, q)) if len(a) else float("nan")
    return dict(dice=float(np.mean(dice)), iou=float(np.mean(iou)),
                d_mean=float(np.mean(derr)) if derr else float("nan"), d_p95=pct(derr, 95), d_max=max(derr) if derr else float("nan"),
                c_mean=float(np.mean(cerr)) if cerr else float("nan"), missed=missed, false_pos=false_pos)


def gflops(net):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        net.eval()
        with FlopCounterMode(display=False) as fc, torch.no_grad():
            net(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE, device=next(net.parameters()).device))
        return fc.get_total_flops() / 1e9
    except Exception:
        return float("nan")


# ---------------------------------------------------------
# 4. TRAIN EACH VARIANT
# ---------------------------------------------------------
trained = {}
for vname in TRAIN_THESE:
    widths = VARIANTS[vname]
    net = LitePupilNet(c=widths).to(DEVICE)
    best_path = "%s/pupil_%s_best.pth" % (WEIGHTS_DIR, vname)
    print("\n=== %s %s | %.2f GFLOPs | %.3f M parameters ==="
          % (vname, widths, gflops(net), sum(p.numel() for p in net.parameters()) / 1e6))

    steps = max(1, len(Xtr) // BATCH_SIZE)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * steps, pct_start=0.15)
    best_dice = -1.0

    for epoch in range(EPOCHS):
        net.train()
        t0 = time.time()
        perm = torch.randperm(len(Xtr), device=Xtr.device)
        run = 0.0
        for s in range(steps):
            idx = perm[s * BATCH_SIZE:(s + 1) * BATCH_SIZE]
            xb = Xtr[idx].to(DEVICE).float() / 255.0
            yb = Ytr[idx].to(DEVICE).float()
            xb, yb = augment(xb, yb)
            logits = net(xb)
            loss = seg_loss(logits, yb)
            if baselines:
                with torch.no_grad():
                    soft = torch.sigmoid(torch.stack([m(xb) for _, m in baselines]).mean(0))
                loss = loss + KD_WEIGHT * F.binary_cross_entropy_with_logits(logits, soft)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step()
            sched.step()
            run += loss.item()

        net.eval()
        d = []
        with torch.no_grad():
            for i in range(0, len(Xva), 16):
                xb = Xva[i:i + 16].to(DEVICE).float() / 255.0
                yb = Yva[i:i + 16].to(DEVICE).float()
                p = (net(xb) > 0).float()
                inter = (p * yb).sum((1, 2, 3))
                d.append(((2 * inter + 1) / (p.sum((1, 2, 3)) + yb.sum((1, 2, 3)) + 1)).cpu())
        vdice = float(torch.cat(d).mean())
        mark = ""
        if vdice > best_dice:
            best_dice = vdice
            torch.save(net.state_dict(), best_path)
            mark = "  <- saved"
        print("Epoch %2d/%d | loss %.4f | val Dice %.4f | %.1fs%s"
              % (epoch + 1, EPOCHS, run / steps, vdice, time.time() - t0, mark))

    net.load_state_dict(torch.load(best_path, map_location=DEVICE))
    trained[vname] = net

# ---------------------------------------------------------
# 5. COMPARISON ON THE TEST SET (what the app cares about)
# ---------------------------------------------------------
print("\nTest set, %d images. Diameter error is |predicted - labelled| pupil diameter in pixels." % len(Xte))
print("%-6s %8s | %6s %6s | %8s %8s %8s | %8s | %6s %6s"
      % ("model", "GFLOPs", "Dice", "IoU", "d mean", "d 95th", "d max", "ctr mean", "missed", "false+"))
rows = [(n, m) for n, m in baselines] + [(n, m) for n, m in trained.items()]
for name, m in rows:
    r = app_eval(m, Xte, Yte)
    print("%-6s %8.2f | %6.4f %6.4f | %8.2f %8.2f %8.2f | %8.2f | %6d %6d"
          % (name, gflops(m), r["dice"], r["iou"], r["d_mean"], r["d_p95"], r["d_max"], r["c_mean"], r["missed"], r["false_pos"]))

# ---------------------------------------------------------
# 6. SHOW ONE TEST IMAGE PER NEW MODEL
# ---------------------------------------------------------
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2 + len(trained), figsize=(4 * (2 + len(trained)), 4))
    ax[0].imshow(Xte[0, 0].numpy(), cmap="gray"); ax[0].set_title("input")
    ax[1].imshow(Yte[0, 0].cpu().numpy(), cmap="gray"); ax[1].set_title("label")
    for k, (name, m) in enumerate(trained.items()):
        with torch.no_grad():
            pm = (m(Xte[:1].to(DEVICE).float() / 255.0) > 0)[0, 0].cpu().numpy()
        ax[2 + k].imshow(pm, cmap="gray"); ax[2 + k].set_title(name)
    for a in ax:
        a.axis("off")
    plt.show()
except Exception as e:
    print("preview skipped:", e)

# ---------------------------------------------------------
# 7. EXPORT ONNX (build the TensorRT engine from these on the Jetson)
#    opset 11, input "input", output "logits": same names the tracker binds.
# ---------------------------------------------------------
for vname, m in trained.items():
    onnx_path = "%s/pupil_%s_%dx%d.onnx" % (WEIGHTS_DIR, vname, IMG_SIZE, IMG_SIZE)
    em = LitePupilNet(c=VARIANTS[vname])
    em.load_state_dict({k: v.cpu() for k, v in m.state_dict().items()})
    em.eval()
    dummy = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE)
    try:
        torch.onnx.export(em, dummy, onnx_path, opset_version=11, input_names=["input"],
                          output_names=["logits"], do_constant_folding=True, dynamo=False)
    except TypeError:
        torch.onnx.export(em, dummy, onnx_path, opset_version=11, input_names=["input"],
                          output_names=["logits"], do_constant_folding=True)
    ops = Counter(n.op_type for n in onnx.load(onnx_path).graph.node)
    print("\n%s -> %s\n  operators: %s" % (vname, onnx_path, dict(ops)))
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        x = Xte[:1].float() / 255.0
        with torch.no_grad():
            ref = em(x).numpy()
        out = sess.run(None, {"input": x.numpy()})[0]
        print("  ONNX vs PyTorch max logit difference: %.2e (should be about 1e-4 or smaller)" % np.abs(ref - out).max())
    except ImportError:
        print("  onnxruntime not installed, check skipped")

print("\nFiles now in %s:" % WEIGHTS_DIR)
for _n in sorted(os.listdir(WEIGHTS_DIR)):
    print("  ", _n, "%.2f MB" % (os.path.getsize(os.path.join(WEIGHTS_DIR, _n)) / 1e6))
print("\nDone. Copy the .onnx you want from %s/ to the Jetson, into the controllers folder." % WEIGHTS_DIR + "")